# Hierarchical Supervisors


Hierarchical Supervisors [Step 06 - Nested Supervisor Trees]

> **MLCourse - Agentic AI - LangGraph**

Hierarchical supervisors extend the flat supervisor pattern into a tree
structure. A top-level supervisor delegates to sub-supervisors, each of
which manages its own team of worker agents. This mirrors real org charts
and scales better than a single flat supervisor with many direct reports.

# What you will learn

1. Building a two-level supervisor hierarchy.
2. How sub-supervisors manage their own teams independently.
3. The top supervisor routes to sub-supervisors, not individual workers.
4. Visualizing the hierarchical graph structure.

### Key takeaways

- Hierarchical supervision reduces the routing burden on any single node.
- Each sub-supervisor can specialize its routing logic for its domain.
- The pattern composes: you can nest supervisors arbitrarily deep.

### Setup: imports, environment


In [ ]:
import os                           # env access
from dotenv import load_dotenv      # .env loading

load_dotenv(override=False)         # load without overriding

from typing import Annotated, TypedDict  # typed state
from langgraph.graph import StateGraph, END  # graph primitives
from langchain_ollama import ChatOllama  # local LLM
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage  # messages
from langchain_core.tools import tool  # tool decorator


### Model guard: check Ollama


In [ ]:
try:                                        # connectivity test
    _test = ChatOllama(model="llama3.1:8b", temperature=0)
    _test.invoke("ping")
    LLM_AVAILABLE = True
    print("Ollama reachable")
except Exception as exc:
    LLM_AVAILABLE = False
    print("Ollama not reachable:", exc)


### Define worker-level tools


In [ ]:
@tool
def code_search(query: str) -> str:
    """Search code repositories for relevant code snippets.

    Args:
        query: The search term to find in code repositories.
    """
    return "Code result: Found 2 implementations matching '%s'" % query

@tool
def write_code(specification: str) -> str:
    """Write code based on a specification.

    Args:
        specification: The coding specification or requirements.
    """
    return "Code written for: %s (3 functions, 45 lines)" % specification

@tool
def documentation_search(query: str) -> str:
    """Search documentation for information.

    Args:
        query: The documentation search query.
    """
    return "Docs result: Found relevant articles for '%s'" % query

@tool
def write_documentation(topic: str) -> str:
    """Write documentation for a given topic.

    Args:
        topic: The topic to document.
    """
    return "Documentation written for: %s (3 sections, 200 words)" % topic


### Define the hierarchical state with step counter


In [ ]:
class HierarchicalState(TypedDict):
    """State carries messages, active node, and step counter."""
    messages: Annotated[list, "conversation messages"]
    active_node: str                      # current active node for routing
    step: int                             # global step counter


### Worker nodes: leaf-level agents


In [ ]:
def coder_worker(state: HierarchicalState) -> dict:
    """Coder worker: writes code based on requests."""
    step = state.get("step", 0) + 1
    if not LLM_AVAILABLE:
        return {"messages": [AIMessage(content="Code written (simulated).")],
                "active_node": "coding_supervisor", "step": step}

    print("[coder] writing code...")
    from langchain.agents import create_agent
    agent = create_agent(
        model=ChatOllama(model="llama3.1:8b", temperature=0),
        tools=[code_search, write_code],
        system_prompt="You are a coder. Search code and write implementations.",
    )
    result = agent.invoke(state)
    msg = result["messages"][-1].content if result.get("messages") else "done"
    print("[coder] done:", msg[:60])
    return {"messages": [AIMessage(content="[coder] " + msg)],
            "active_node": "coding_supervisor", "step": step}

def tester_worker(state: HierarchicalState) -> dict:
    """Tester worker: tests code and finds bugs."""
    step = state.get("step", 0) + 1
    if not LLM_AVAILABLE:
        return {"messages": [AIMessage(content="Tests passed (simulated).")],
                "active_node": "coding_supervisor", "step": step}

    print("[tester] running tests...")
    llm = ChatOllama(model="llama3.1:8b", temperature=0)
    msgs = [SystemMessage(content="You are a code tester. Report test results.")] + state["messages"]
    response = llm.invoke(msgs)
    print("[tester] done:", response.content[:60])
    return {"messages": [AIMessage(content="[tester] " + response.content)],
            "active_node": "coding_supervisor", "step": step}

def researcher_worker(state: HierarchicalState) -> dict:
    """Researcher worker: finds information in docs."""
    step = state.get("step", 0) + 1
    if not LLM_AVAILABLE:
        return {"messages": [AIMessage(content="Research complete (simulated).")],
                "active_node": "docs_supervisor", "step": step}

    print("[researcher] searching docs...")
    from langchain.agents import create_agent
    agent = create_agent(
        model=ChatOllama(model="llama3.1:8b", temperature=0),
        tools=[documentation_search],
        system_prompt="You are a researcher. Search documentation for answers.",
    )
    result = agent.invoke(state)
    msg = result["messages"][-1].content if result.get("messages") else "done"
    print("[researcher] done:", msg[:60])
    return {"messages": [AIMessage(content="[researcher] " + msg)],
            "active_node": "docs_supervisor", "step": step}

def tech_writer_worker(state: HierarchicalState) -> dict:
    """Technical writer worker: creates documentation."""
    step = state.get("step", 0) + 1
    if not LLM_AVAILABLE:
        return {"messages": [AIMessage(content="Documentation written (simulated).")],
                "active_node": "docs_supervisor", "step": step}

    print("[tech_writer] writing docs...")
    from langchain.agents import create_agent
    agent = create_agent(
        model=ChatOllama(model="llama3.1:8b", temperature=0),
        tools=[write_documentation],
        system_prompt="You are a technical writer. Create clear documentation.",
    )
    result = agent.invoke(state)
    msg = result["messages"][-1].content if result.get("messages") else "done"
    print("[tech_writer] done:", msg[:60])
    return {"messages": [AIMessage(content="[tech_writer] " + msg)],
            "active_node": "docs_supervisor", "step": step}


### Sub-supervisor nodes: manage worker teams


In [ ]:
#
# A practical guardrail: a small local model will sometimes answer "FINISH" on
# the very first turn, before any worker has produced anything. We keep the LLM
# in charge of *which* worker runs, but refuse to let it close out a team that
# has not done any work yet. `worker_has_run` inspects the conversation for a
# tagged worker message, so the rule is explicit rather than hidden.

MAX_STEPS = 4   # hard cap on worker turns per team, so the loop always ends

def worker_has_run(state: HierarchicalState, tags: tuple) -> bool:
    """True if any worker on this team has already posted a tagged message."""
    for m in state.get("messages", []):
        content = getattr(m, "content", "") or ""
        if any(content.startswith(t) for t in tags):
            return True
    return False


def coding_supervisor(state: HierarchicalState) -> dict:
    """Coding sub-supervisor: routes between coder and tester workers."""
    step = state.get("step", 0)
    if LLM_AVAILABLE:
        llm = ChatOllama(model="llama3.1:8b", temperature=0)
        sys_prompt = (
            "You manage a coding team with two workers:\n"
            "- coder: writes code\n"
            "- tester: tests code\n"
            "- FINISH: when the coding task is complete\n"
            "Decide which worker should act next. Reply with just the name."
        )
        msgs = [SystemMessage(content=sys_prompt)] + state["messages"]
        response = llm.invoke(msgs)
        decision = response.content.strip().lower()
        if "coder" in decision:
            next_node = "coder"
        elif "tester" in decision:
            next_node = "tester"
        else:
            next_node = "FINISH"
    else:
        # Fixed demo: coder -> tester -> FINISH
        sub_step = state.get("step", 0)
        demo_seq = ["coder", "tester", "FINISH"]
        next_node = demo_seq[min(sub_step, len(demo_seq) - 1)]

    # Guardrail: cap the loop so a chatty model cannot spin forever.
    if step >= MAX_STEPS:
        next_node = "FINISH"

    # Guardrail: never finish before this team has produced anything.
    if next_node == "FINISH" and not worker_has_run(state, ("[coder]", "[tester]")):
        print("[coding_supervisor] guardrail: no worker output yet -> 'coder'")
        next_node = "coder"

    print("[coding_supervisor] routing to:", next_node)
    return {"active_node": next_node, "step": step}


def docs_supervisor(state: HierarchicalState) -> dict:
    """Docs sub-supervisor: routes between researcher and tech writer."""
    step = state.get("step", 0)
    if LLM_AVAILABLE:
        llm = ChatOllama(model="llama3.1:8b", temperature=0)
        sys_prompt = (
            "You manage a documentation team with two workers:\n"
            "- researcher: searches for information\n"
            "- tech_writer: writes documentation\n"
            "- FINISH: when the docs task is complete\n"
            "Decide which worker should act next. Reply with just the name."
        )
        msgs = [SystemMessage(content=sys_prompt)] + state["messages"]
        response = llm.invoke(msgs)
        decision = response.content.strip().lower()
        if "research" in decision:
            next_node = "researcher"
        elif "writer" in decision or "tech" in decision:
            next_node = "tech_writer"
        else:
            next_node = "FINISH"
    else:
        # Fixed demo: researcher -> tech_writer -> FINISH
        sub_step = state.get("step", 0)
        demo_seq = ["researcher", "tech_writer", "FINISH"]
        next_node = demo_seq[min(sub_step, len(demo_seq) - 1)]

    # Guardrail: cap the loop so a chatty model cannot spin forever.
    if step >= MAX_STEPS:
        next_node = "FINISH"

    # Guardrail: never finish before this team has produced anything.
    if next_node == "FINISH" and not worker_has_run(state, ("[researcher]", "[tech_writer]")):
        print("[docs_supervisor] guardrail: no worker output yet -> 'researcher'")
        next_node = "researcher"

    print("[docs_supervisor] routing to:", next_node)
    return {"active_node": next_node, "step": step}


### Top supervisor: routes between sub-supervisors


In [ ]:
def top_supervisor(state: HierarchicalState) -> dict:
    """Top-level supervisor: decides between coding and docs sub-supervisors."""
    step = state.get("step", 0) + 1
    if LLM_AVAILABLE:
        llm = ChatOllama(model="llama3.1:8b", temperature=0)
        sys_prompt = (
            "You are the top-level supervisor. You manage two sub-supervisors:\n"
            "- coding_supervisor: handles all coding tasks\n"
            "- docs_supervisor: handles all documentation tasks\n"
            "- FINISH: when the overall task is complete\n"
            "Decide which sub-supervisor should act next. Reply with just the name."
        )
        msgs = [SystemMessage(content=sys_prompt)] + state["messages"]
        response = llm.invoke(msgs)
        decision = response.content.strip().lower()
        if "coding" in decision or "code" in decision:
            next_node = "coding_supervisor"
        elif "doc" in decision:
            next_node = "docs_supervisor"
        else:
            next_node = "FINISH"
    else:
        # Fixed demo: coding -> docs -> FINISH
        demo_seq = ["coding_supervisor", "docs_supervisor", "FINISH"]
        idx = min(step - 1, len(demo_seq) - 1)
        next_node = demo_seq[idx]

    print("[top_supervisor] routing to:", next_node, "(step %d)" % step)
    return {"active_node": next_node, "step": step}


### Routing functions for each supervisor level


In [ ]:
def route_top(state: HierarchicalState) -> str:
    """Route from top supervisor to sub-supervisors."""
    return state.get("active_node", "FINISH")

def route_coding(state: HierarchicalState) -> str:
    """Route from coding supervisor to workers."""
    return state.get("active_node", "FINISH")

def route_docs(state: HierarchicalState) -> str:
    """Route from docs supervisor to workers."""
    return state.get("active_node", "FINISH")


### Build the hierarchical graph


In [ ]:
builder = StateGraph(HierarchicalState)

# Top level
builder.add_node("top_supervisor", top_supervisor)

# Sub-supervisors
builder.add_node("coding_supervisor", coding_supervisor)
builder.add_node("docs_supervisor", docs_supervisor)

# Workers
builder.add_node("coder", coder_worker)
builder.add_node("tester", tester_worker)
builder.add_node("researcher", researcher_worker)
builder.add_node("tech_writer", tech_writer_worker)

# Entry point
builder.set_entry_point("top_supervisor")

# Top supervisor -> sub-supervisors or END
builder.add_conditional_edges(
    "top_supervisor", route_top,
    {
        "coding_supervisor": "coding_supervisor",
        "docs_supervisor": "docs_supervisor",
        "FINISH": END,
    },
)

# Coding supervisor -> workers
builder.add_conditional_edges(
    "coding_supervisor", route_coding,
    {
        "coder": "coder",
        "tester": "tester",
        "FINISH": END,
    },
)

# Docs supervisor -> workers
builder.add_conditional_edges(
    "docs_supervisor", route_docs,
    {
        "researcher": "researcher",
        "tech_writer": "tech_writer",
        "FINISH": END,
    },
)

# Workers return to their respective sub-supervisors
builder.add_edge("coder", "coding_supervisor")
builder.add_edge("tester", "coding_supervisor")
builder.add_edge("researcher", "docs_supervisor")
builder.add_edge("tech_writer", "docs_supervisor")

graph = builder.compile()

print("Hierarchical graph compiled:")
print("  top_supervisor -> {coding_supervisor, docs_supervisor}")
print("  coding_supervisor -> {coder, tester}")
print("  docs_supervisor -> {researcher, tech_writer}")


### Visualize the hierarchical graph


In [ ]:
from IPython.display import Image, display

try:                                        # wrap in try/except for offline
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("Visualization unavailable:", exc)
    print("Nodes:", list(builder.nodes.keys()))


### Run the hierarchy: coding + docs task


In [ ]:
print("=== Hierarchical execution: coding + docs task ===")
print()

initial = {
    "messages": [HumanMessage(content="Write a Python function and document it.")],
    "active_node": "coding_supervisor",
    "step": 0,
}

for step in graph.stream(initial):
    for node_name, output in step.items():
        active = output.get("active_node", "")
        msgs = output.get("messages", [])
        if msgs:
            for m in msgs:
                content = m.content if hasattr(m, "content") else str(m)
                print("[%s] %s" % (node_name, content[:70]))
        else:
            print("[%s] -> %s" % (node_name, active))

print()
print("NOTEBOOK COMPLETE: hierarchical supervisors demonstrated successfully")
